In [46]:
from pathlib import Path
from metasmith.python_api import Agent, Source, Std, DataInstanceLibrary, TransformInstanceLibrary, Endpoint

dtypes, containers, transforms = Std()

In [47]:
path_to_agent_home = Path("./cache/local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
smith.Deploy()

2025-08-05_20-08-13  | >>> AGENT_HOME=/home/tony/workspace/tools/Metasmith/main/tests/cache/local_home
2025-08-05_20-08-13  | >>> mkdir -p $AGENT_HOME
2025-08-05_20-08-13  | >>> mkdir -p /home/tony/.globus
2025-08-05_20-08-13  | >>> mkdir -p /home/tony/.globusonline
2025-08-05_20-08-13  | >>> {if not exists}: apptainer pull/metasmith.sif docker://quay.io/hallamlab/metasmith:latest
2025-08-05_20-08-13  | staged [msm_stub]
2025-08-05_20-08-13  | staged [msm]
2025-08-05_20-08-13  | staged [lib/agent.yml]
2025-08-05_20-08-13  | staged [lib/msm_bootstrap]
2025-08-05_20-08-13  | staged [lib/nextflow_config]
2025-08-05_20-08-13  | deploying [5] staged files
2025-08-05_20-08-13  | >>> cd /home/tony/workspace/tools/Metasmith/main/tests/cache/local_home && ./msm api deploy_from_container
2025-08-05_20-08-13  | including dev binds
2025-08-05_20-08-14  | 2025-08-05_20-08-14  | api call to [deploy_from_container] with [{}]
2025-08-05_20-08-14  | 2025-08-05_20-08-14  | deploying to [/ws]
2025-08-05_

In [48]:
from metasmith.coms.ipc import LiveShell
from local.constants import WORKSPACE_ROOT

home = f"{WORKSPACE_ROOT}/main/tests/cache/local_home"

with LiveShell() as shell:
    shell.RegisterOnOut(lambda x: print(x))
    shell.RegisterOnErr(lambda x: print(f"E: {x}"))
    # shell.Exec(f"rsync -ac --progress --mkpath {WORKSPACE_ROOT}/metasmith.sif {home}/metasmith.sif")
    shell.Exec(f"rsync -acu --progress --mkpath {WORKSPACE_ROOT}/main/relay_agent/dist/relay {home}/relay/msm_relay")
    shell.Exec(f"rsync -acu --progress --mkpath --exclude=__pycache__ {WORKSPACE_ROOT}/src/metasmith/ {home}/dev/metasmith")
    shell.Exec(f"rsync -acu --progress --mkpath --exclude=__pycache__ {WORKSPACE_ROOT}/src/metasmith/nextflow_config {home}/lib/")


sending incremental file list
sending incremental file list
std/std_containers.xgdb/_metadata/index.yml
             44 100%    0.00kB/s    0:00:00 (xfr#1, to-chk=35/139)
sending incremental file list


In [49]:
# # generate transform lib template
# transforms = TransformInstanceLibrary("./exec_with_container.xgdb")
# transforms.AddStub("mre.py")
# transforms.Save()

transforms = TransformInstanceLibrary.Load("./exec_with_container.xgdb")
for _path, _, tr in transforms.IterateTransforms():
    print(tr.name)
    for p in tr.model.requires:
        print(" ", p)
    print("->")
    for p in tr.model.produces:
        print(" ", p)

mre
  (D:{"data":"OCI"}-{"format":"Software container"}-{"provides":"GNU coreutils"})
  (D:{"data":"OCI"}-{"format":"Software container"}-{"provides":"flye"})
->
  (D:{"data":"Sequence assembly"})


In [50]:
task = smith.GenerateWorkflow(
    given=[containers],
    transforms=[transforms],
    targets=[dtypes.types["assembly"]]
)
for x in task.plan.steps:
    print(x.transform.name)

mre


In [51]:
smith.StageWorkflow(task, on_exist="clear")
smith.RunWorkflow(task)

2025-08-05_20-08-17  | connecting to deployed agent
2025-08-05_20-08-17  | starting relay service


E| > 2025-08-05_20-08-18 E| relay server already running in [relay/connections]


 | > 2025-08-05_20-08-18  | connecting to relay as [TZY5D9Y7crvT]
2025-08-05_20-08-18  | sending metadata for workflow [ECc4g5fv]
2025-08-05_20-08-20  | staging
 | > including dev binds
 | > 2025-08-05_20-08-21  | api call to [stage_workflow] with [{'task_key': 'ECc4g5fv'}]
 | > 2025-08-05_20-08-21  | staging workflow [ECc4g5fv] with [1] data libs and [1] transform libs
 | > 2025-08-05_20-08-21  | ex| /home/tony/workspace/tools/Metasmith/main/tests/cache/local_home
 | > 2025-08-05_20-08-21  | work [/ws/runs/ECc4g5fv]
 | > 2025-08-05_20-08-21  | data [/msm_home/data]
 | > 2025-08-05_20-08-21  | external work [/home/tony/workspace/tools/Metasmith/main/tests/cache/local_home/runs/ECc4g5fv]
 | > 2025-08-05_20-08-21  | external data [/home/tony/workspace/tools/Metasmith/main/tests/cache/local_home/data]
 | > 2025-08-05_20-08-21  | moving remote data libraries to [/msm_home/data]
 | > 2025-08-05_20-08-21  | using nextflow preset [default]
 | > 2025-08-05_20-08-21  | [ECc4g5fv] staged to [{AG

E| > 2025-08-05_20-08-22 E| relay server already running in [relay/connections]


2025-08-05_20-08-22  | triggering execution of [ECc4g5fv]
2025-08-05_20-08-23  | closing connection


In [44]:
with LiveShell() as shell:
    r = shell.Exec("echo asdf", history=True)
    print(r.out)

['asdf']
